In [ ]:
!lamin disconnect
!lamin login testuser1
!lamin delete --force test-init-cloud-with-db
!docker stop pgtest && docker rm pgtest

In [ ]:
import os
import laminci
import lamindb_setup as ln_setup
from lamindb_setup.core._hub_core import delete_instance
from lamindb_setup.core._hub_crud import select_instance_by_id
from lamindb_setup.core._hub_client import call_with_fallback_auth

In [ ]:
storage = f"s3://lamindb-ci/cloud_with_db_{os.getenv('LAMIN_ENV', 'prod')}"

In [ ]:
pgurl = laminci.db.setup_local_test_postgres()

Test initializing an instance with a cloud storage and a postgres db url.

In [ ]:
ln_setup.init(storage=storage, name="test-init-cloud-with-db", db=pgurl)

In [ ]:
from lamindb import __version__ as lamindb_version

instance_record = call_with_fallback_auth(
    select_instance_by_id, instance_id=ln_setup.settings.instance._id.hex
)

assert instance_record["lamindb_version"] == lamindb_version

In [ ]:
assert ln_setup.settings.instance.name == "test-init-cloud-with-db"

In [ ]:
root = ln_setup.settings.storage.root
mark_file = ln_setup.settings.storage._mark_storage_root

In [ ]:
assert mark_file.exists()

In [ ]:
# this is a postgres instance, it doesn't have sqlite file
# we test here that delete_instance properly cleans up mark files if delete_mark_files=True
delete_instance(
    "testuser1/test-init-cloud-with-db", require_empty=True, delete_mark_files=True
)

In [ ]:
assert not mark_file.exists()
assert not root.exists()

In [ ]:
!docker stop pgtest && docker rm pgtest